<a href="https://colab.research.google.com/github/acamogh/Gradio_Quiz_App/blob/main/Google_Cloud_Generative_AI_Leader_Practice_Exam_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# =====================================================================
# CELL 1: ENVIRONMENT CONFIGURATION & PARSING (UPDATED)
# =====================================================================
print("📦 Installing mobile packages... (This takes about 40 seconds)")
import sys
import subprocess
import os
import glob
import json
import random  # Imported for random chunk selection later

# Install clean standalone packages
subprocess.run([sys.executable, "-m", "pip", "install", "--ignore-installed", "-q", "-U", "groq"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "langchain-community", "pypdf", "tiktoken", "gradio"], check=True)
print("✅ Libraries installed successfully.")

import gradio as gr
from groq import Groq
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader

print("\n📂 Scanning notebook root for PDF files...")
pdf_files = glob.glob("*.pdf")

if not pdf_files:
    raise FileNotFoundError("⚠️ No PDF files found! Tap the folder icon on the left of Colab, upload your 8 PDFs, and run this cell again.")

print(f"📄 Found {len(pdf_files)} study guide files. Extracting content text...")

# CHANGED: We now store text in a list of chunks instead of one giant string
KNOWLEDGE_CHUNKS = []

for pdf in pdf_files:
    try:
        loader = PyPDFLoader(pdf)
        pages = loader.load()
        for page in pages:
            text = page.page_content.strip()
            if text:
                # Append each page as its own context chunk
                KNOWLEDGE_CHUNKS.append({
                    "source": pdf,
                    "content": text
                })
    except Exception as e:
        print(f"   Skipping or error reading {pdf}: {e}")

print(f"✅ Setup Complete. Loaded {len(KNOWLEDGE_CHUNKS)} individual pages into memory matrix.")

📦 Installing mobile packages... (This takes about 40 seconds)
✅ Libraries installed successfully.

📂 Scanning notebook root for PDF files...
📄 Found 8 study guide files. Extracting content text...
✅ Setup Complete. Loaded 588 individual pages into memory matrix.


In [14]:
# =====================================================================
# CELL 2: STATE ENGINE & TARGETED GRADIO CANVAS 111
# =====================================================================
import random
import json

class MobileQuizEngine:
    def __init__(self, knowledge_chunks):
        self.chunks = knowledge_chunks
        self.incorrect_log = []
        self.current_quiz = None

        # High Volume Strategy Configuration
        self.model_id = 'openai/gpt-oss-120b'
        self.backup_model_id = 'openai/gpt-oss-20b'
        self.active_model_used = "None"

        try:
            api_key = userdata.get('GROQ_API_KEY')
            self.groq_client = Groq(api_key=api_key)
        except Exception:
            raise ValueError("❌ GROQ_API_KEY missing. Set it in Colab's Secrets menu (Key Icon).")

    def generate_question(self):
        review_context = ""
        if self.incorrect_log and len(self.incorrect_log) % 3 == 0:
            past_mistake = self.incorrect_log[-1]
            review_context = f"""
            CRITICAL REVIEW CONTEXT: The student previously failed a question matching this target concept: '{past_mistake['concept']}'.
            Do NOT reuse this old question text: '{past_mistake['question_text']}'.
            Instead, heavily paraphrase the scenario, alter the variables entirely, and address the concept from a completely different angle.
            """

        selected_chunks = random.sample(self.chunks, min(3, len(self.chunks)))

        dynamic_context = ""
        for idx, chunk in enumerate(selected_chunks):
            dynamic_context += f"\n--- Context Source {idx+1}: File '{chunk['source']}', Page {chunk['page']} ---\n"
            dynamic_context += chunk['content'] + "\n"

        system_instruction = f"""
        You are an expert exam simulator for the Google Cloud Generative AI Leader certification.

        Knowledge Base Material with Reference Sources:
        {dynamic_context}

        Instructions:
        1. NEVER copy exact sentences or verbatim questions from the source text.
        2. Generate one highly challenging multiple-choice question matching the official certification level.
        3. Provide exactly four options (A, B, C, D) with strictly ONE correct answer.
        4. Do NOT reveal clues or answers inside your fields.
        5. Identify which specific source file and page number you primarily based the question on.

        {review_context}

        Output Structure Requirement: Return data STRICTLY inside a standard JSON object matching this structural layout:
        {{
            "question": "Clear scenario/question text goes here...",
            "options": {{
                "A": "Option text content",
                "B": "Option text content",
                "C": "Option text content",
                "D": "Option text content"
            }},
            "correct_answer": "A, B, C, or D",
            "concept": "The underlying technical principle tested",
            "explanation": "Detailed professional rationale breakdown describing why the choice is correct and others are wrong.",
            "source_file": "The exact name of the PDF file used",
            "source_page": "The exact page number used as an integer or string"
        }}
        """

        try:
            response = self.groq_client.chat.completions.create(
                model=self.model_id,
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": "Generate the next certification exam question based on the provided context."}
                ],
                response_format={"type": "json_object"},
                temperature=0.8
            )
            self.active_model_used = "🤖 Premium Engine (gpt-oss-120b)"

        except Exception as e:
            print(f"⚡ Primary model rate-limit/timeout hit. Switching to high-volume gpt-oss-20b... Details: {e}")
            response = self.groq_client.chat.completions.create(
                model=self.backup_model_id,
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": "Generate the next certification exam question based on the provided context."}
                ],
                response_format={"type": "json_object"},
                temperature=0.7
            )
            self.active_model_used = "⚡ Fast Fallback Workhorse (gpt-oss-20b)"

        try:
            raw_content = response.choices[0].message.content
            self.current_quiz = json.loads(raw_content)
            return self.current_quiz
        except Exception as e:
            print(f"Error parsing JSON output: {e}")
            return None

    def log_failure(self):
        if self.current_quiz:
            file_ref = self.current_quiz.get("source_file", "Unknown")
            page_ref = self.current_quiz.get("source_page", "?")
            self.incorrect_log.append({
                "concept": self.current_quiz.get("concept"),
                "question_text": self.current_quiz.get("question"),
                "source": f"{file_ref} (Pg. {page_ref})"
            })

    def get_summary(self):
        if not self.incorrect_log:
            return "🎉 Perfect record! No weak areas logged yet."
        summary = "### 📊 Weak Concepts to Review:\n"
        for idx, item in enumerate(self.incorrect_log, 1):
            summary += f"{idx}. **Domain Focus**: {item['concept']} *(Source: {item['source']})*\n"
        return summary

# Instantiate active engine links
quiz_engine = MobileQuizEngine(KNOWLEDGE_CHUNKS)

# =====================================================================
# GRADIO INTERFACE FUNCTIONS
# =====================================================================
def load_new_question():
    data = quiz_engine.generate_question()
    if not data:
        # Returns an empty string fallback for the new metadata tracker panel
        return "❌ API Timeout. Click 'Fetch Next Question' to retry.", [], "", ""

    # Isolate question text body
    q_text = f"### Question:\n{data['question']}"

    # CHANGED: Create the dynamic reference row cleanly as an isolated string
    file_name = data.get('source_file', 'Unknown PDF')
    page_num = data.get('source_page', 'Unknown Page')
    meta_text = f"📖 **Source:** `{file_name}` (Page `{page_num}`) | 🧠 **Model In Use:** `{quiz_engine.active_model_used}`"

    choices_list = [
        f"A: {data['options'].get('A', '')}",
        f"B: {data['options'].get('B', '')}",
        f"C: {data['options'].get('C', '')}",
        f"D: {data['options'].get('D', '')}"
    ]
    # Return 4 values to map directly to the expanded UI targets
    return q_text, gr.Radio(choices=choices_list, value=None), "", meta_text

def evaluate_selection(choice):
    if not quiz_engine.current_quiz:
        return "⚠️ Please load a question first."
    if not choice:
        return "⚠️ Please click an option bubble before submitting your selection."

    user_letter = choice.split(":")[0].strip().upper()
    correct_letter = quiz_engine.current_quiz['correct_answer'].strip().upper()

    if user_letter == correct_letter:
        score_text = "### ✅ CORRECT!\n\n"
    else:
        file_name = quiz_engine.current_quiz.get('source_file', 'Unknown PDF')
        page_num = quiz_engine.current_quiz.get('source_page', 'Unknown Page')
        score_text = f"### ❌ INCORRECT.\nThe correct answer choice was **({correct_letter})**.\n\n📍 *Double check this on page {page_num} of {file_name}*\n\n"
        quiz_engine.log_failure()

    return f"{score_text}📝 **Rationale Solution Breakdown**:\n{quiz_engine.current_quiz['explanation']}"

def view_summary_panel():
    return quiz_engine.get_summary()

# =====================================================================
# APPLICATION INTERFACE GENERATION CANVAS
# =====================================================================
with gr.Blocks(theme=gr.themes.Soft()) as mobile_app:
    gr.Markdown("# 🧠 Google Cloud GenAI Leader App Hub")

    with gr.Group():
        question_display = gr.Markdown("### Tap '➡️ Fetch Next Question' at the bottom to begin your practice run!")

    choice_radio = gr.Radio(choices=[], label="👇 Select your answer option:")
    submit_btn = gr.Button("🔒 Submit Selection", variant="primary")
    feedback_display = gr.Markdown("")

    # Pinned navigation controls layout row panel matrix
    with gr.Row():
        next_btn = gr.Button("➡️ Fetch Next Question", variant="secondary")
        summary_btn = gr.Button("📊 Weak Summary Dashboard", variant="stop")

    # CHANGED: Placed the tracking block explicitly BELOW the navigation row canvas
    metadata_tracker = gr.Markdown("")

    # Bind backend method updates sequentially to handle event changes smoothly
    next_btn.click(
        fn=load_new_question,
        inputs=[],
        # CHANGED: Added metadata_tracker as the fourth output channel parameter target
        outputs=[question_display, choice_radio, feedback_display, metadata_tracker],
        queue=True
    )

    submit_btn.click(fn=evaluate_selection, inputs=[choice_radio], outputs=[feedback_display])
    summary_btn.click(fn=view_summary_panel, inputs=[], outputs=[feedback_display])

# Run app interface
mobile_app.launch(inline=False, share=True, debug=False)

/tmp/ipykernel_4435/755544499.py:178: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as mobile_app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e461704fea3489ed7a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
